# k-means：衣類・靴・バッグ

## 1. データセットを確認する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits

SEED = 42

#@title Fashion-MNISTを読み込む
from pathlib import Path
import gzip
import hashlib
import shutil
import struct
import urllib.request

# 出典: https://github.com/zalandoresearch/fashion-mnist
CACHE = Path("kmeans_data")
CACHE.mkdir(exist_ok=True)
BASE_URLS = [
    "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/",
    "https://storage.googleapis.com/tensorflow/tf-keras-datasets/",
]

def load_idx(filename, checksum):
    path = CACHE / filename
    valid = lambda p: p.is_file() and hashlib.md5(p.read_bytes()).hexdigest() == checksum
    if not valid(path):
        last_error = None
        for base in BASE_URLS:
            temp = path.with_suffix(path.suffix + ".part")
            try:
                with urllib.request.urlopen(base + filename, timeout=60) as src, temp.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
                if not valid(temp):
                    raise ValueError("ダウンロードしたファイルが不完全です。")
                temp.replace(path)
                break
            except Exception as exc:
                last_error = exc
                temp.unlink(missing_ok=True)
        else:
            raise RuntimeError(f"取得できませんでした。接続を確認してください: {filename}") from last_error
    with gzip.open(path, "rb") as f:
        zero, dtype_code, ndim = struct.unpack(">HBB", f.read(4))
        shape = struct.unpack(">" + "I" * ndim, f.read(4 * ndim))
        values = np.frombuffer(f.read(), dtype=np.uint8)
    if zero != 0 or dtype_code != 8 or values.size != np.prod(shape):
        raise ValueError("IDXファイルの形式が不正です。")
    return values.reshape(shape)

all_images = load_idx("train-images-idx3-ubyte.gz", "8d4fb7e6c68d591d4c3dfef9ec88bf0d")
all_labels = load_idx("train-labels-idx1-ubyte.gz", "25c81989df183df01b3e8a0aad5dffbe")
if all_images.shape != (60000, 28, 28) or all_labels.shape != (60000,):
    raise ValueError("Fashion-MNISTの画像数またはサイズが想定と異なります。")
if np.any(all_labels > 9):
    raise ValueError("正解ラベルが0〜9の範囲外です。")

N_SAMPLES = 6000  # 重い場合は3000などに変更
if not 100 <= N_SAMPLES <= len(all_images):
    raise ValueError(f"N_SAMPLESは100〜{len(all_images)}にしてください。")

# 正解ラベルを見ずに画像を抽出
selected = np.random.default_rng(SEED).choice(len(all_images), N_SAMPLES, replace=False)
images = all_images[selected]
y_true = all_labels[selected]
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
PIXEL_MAX = 255.0

print("画像:", images.shape, "/ 正解ラベル:", y_true.shape)
print("画素値:", images.min(), "〜", images.max())

# 各正解カテゴリから1枚ずつ閲覧
rng = np.random.default_rng(SEED)
preview_ids = [rng.choice(np.flatnonzero(y_true == c))
               for c in range(len(class_names)) if np.any(y_true == c)]

fig, axes = plt.subplots(2, 5, figsize=(11, 5), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, i in zip(axes.flat, preview_ids):
    ax.imshow(images[i], cmap="gray", vmin=0, vmax=PIXEL_MAX)
    ax.set_title(f"{class_names[int(y_true[i])]}\nindex={i}", fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
INDEX = 2  # この番号を変えると別の画像を確認できる
if not 0 <= INDEX < len(images):
    raise ValueError(f"INDEXは0〜{len(images) - 1}にしてください。")

plt.figure(figsize=(3.5, 3.5))
plt.imshow(images[INDEX], cmap="gray", vmin=0, vmax=PIXEL_MAX)
plt.title(f"index={INDEX} / true={class_names[int(y_true[INDEX])]}")
plt.axis("off")
plt.show()

## 2. 前処理とk-means

In [ ]:
# 画像を1行に並べ、画素値を0〜1にそろえる。画素ごとの標準化・次元削減はしない。
X = images.reshape(len(images), -1).astype(np.float32) / PIXEL_MAX
print("k-meansへの入力:", X.shape, "（画像数, 変数数）")

K = 10
N_INIT = 10

def run_kmeans(X, k, seed=SEED):
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(X):
        raise ValueError("kは1〜画像数の整数にしてください。")
    model = KMeans(n_clusters=k, init="k-means++", n_init=N_INIT,
                   algorithm="lloyd", random_state=seed)
    with threadpool_limits(limits=2):  # CPUスレッド数を制限
        cluster_id = model.fit_predict(X)  # 正解ラベルは渡さない
    return model, cluster_id

model, cluster_id = run_kmeans(X, K)
print(f"k={model.n_clusters} / 変数数={X.shape[1]} / 反復回数={model.n_iter_}")

## 3. クラスタの平均画像とサンプル

In [ ]:
#@title クラスタの画像表示
def show_clusters(images, cluster_id, k, n_examples=4, seed=SEED):
    if not isinstance(n_examples, (int, np.integer)) or n_examples < 1:
        raise ValueError("n_examplesは1以上の整数にしてください。")
    rng = np.random.default_rng(seed)
    for first in range(0, k, 5):  # 1図あたり最大5クラスタ
        cluster_numbers = list(range(first, min(first + 5, k)))
        fig, axes = plt.subplots(len(cluster_numbers), n_examples + 1,
                                 figsize=(2 * (n_examples + 1), 1.9 * len(cluster_numbers)),
                                 squeeze=False)
        for row, c in enumerate(cluster_numbers):
            for ax in axes[row]:
                ax.axis("off")
            members = np.flatnonzero(cluster_id == c)
            if len(members) == 0:
                axes[row, 0].set_title(f"C{c}: empty")
                continue
            # 所属画像の平均と、重複なしで選んだ実画像
            mean_image = images[members].mean(axis=0)
            axes[row, 0].imshow(mean_image, cmap="gray", vmin=0, vmax=PIXEL_MAX)
            axes[row, 0].set_title(f"C{c}: mean / n={len(members)}", fontsize=10)
            chosen = rng.choice(members, min(n_examples, len(members)), replace=False)
            for col, i in enumerate(chosen, start=1):
                axes[row, col].imshow(images[i], cmap="gray", vmin=0, vmax=PIXEL_MAX)
                axes[row, col].set_title(f"C{c}: index={i}", fontsize=9)
        fig.suptitle(f"k={k} | Mean + {n_examples} random examples", fontsize=13)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        plt.show()

N_EXAMPLES = 4  # 3に変更してもよい
show_clusters(images, cluster_id, model.n_clusters, n_examples=N_EXAMPLES)

## 4. 正解ラベルとの比較

In [ ]:
#@title 正解ラベルとクラスタの比較
from scipy.optimize import linear_sum_assignment

def aligned_counts(y_true, cluster_id, k):
    counts = np.zeros((len(class_names), k), dtype=int)
    np.add.at(counts, (y_true, cluster_id), 1)

    # 1対1の対応で一致数を最大化。kが10以外のときは未対応分を末尾へ。
    matched_rows, matched_cols = linear_sum_assignment(counts, maximize=True)
    rows = list(matched_rows) + [r for r in range(len(class_names)) if r not in matched_rows]
    cols = list(matched_cols) + [c for c in range(k) if c not in matched_cols]
    return counts[np.ix_(rows, cols)], rows, cols

def show_heatmap(y_true, cluster_id, k):
    counts, rows, cols = aligned_counts(y_true, cluster_id, k)
    fig, ax = plt.subplots(figsize=(max(7, 0.62 * k + 2.5), 7))
    im = ax.imshow(counts, cmap="Blues", aspect="auto", vmin=0)
    ax.set_xticks(range(k), [f"C{c}" for c in cols], rotation=45, ha="right")
    ax.set_yticks(range(len(rows)), [class_names[r] for r in rows])
    ax.set_xlabel("Cluster ID (reordered)")
    ax.set_ylabel("True label")
    ax.set_title(f"True label vs cluster | k={k} | same data used for clustering")
    for row in range(counts.shape[0]):
        for col in range(counts.shape[1]):
            value = counts[row, col]
            if value:
                text_color = "white" if value > counts.max() * 0.5 else "black"
                ax.text(col, row, str(value), ha="center", va="center",
                        color=text_color, fontsize=9)
    fig.colorbar(im, ax=ax, label="Number of images")
    fig.tight_layout()
    plt.show()
    return counts

In [ ]:
# 学習に使った画像の事後比較。並べ替えるのは表示順だけで、所属は変えない。
heatmap_counts = show_heatmap(y_true, cluster_id, model.n_clusters)

## 5. PCAによる可視化（任意）

In [ ]:
from sklearn.decomposition import PCA

# 表示専用。k-meansには元の画素値を使っており、この節は丸ごと削除できる。
pca_view = PCA(n_components=2, svd_solver="randomized", random_state=SEED)
with threadpool_limits(limits=2):
    Z_view = pca_view.fit_transform(X)
print(f"2次元で表している分散の割合: {pca_view.explained_variance_ratio_.sum():.1%}")

In [ ]:
# 同じ座標を、クラスタと正解ラベルでそれぞれ色分け
views = [
    (cluster_id, [f"C{c}" for c in range(model.n_clusters)], "Clusters"),
    (y_true, class_names, "True labels"),
]
for labels, names, title in views:
    fig, ax = plt.subplots(figsize=(8, 6))
    for label, name in enumerate(names):
        take = labels == label
        ax.scatter(Z_view[take, 0], Z_view[take, 1], s=10, alpha=0.55, label=name)
    ax.set_xlabel(f"PC1 ({pca_view.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca_view.explained_variance_ratio_[1]:.1%})")
    ax.set_title(f"PCA: display only | {title}")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
    fig.tight_layout()
    plt.show()